# Doc render farm (Blender 5.2 on Kaggle GPU)
Settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On**.
Add the dataset with `doc_walk_packed.blend`, `kaggle_render.py`, `post.py` (Add Input).
Then **Run All**. For long renders use **Save Version → Save & Run All** so it keeps going after you close the tab.
Download `doc_render.mp4` and `frames.zip` from the Output tab when done.

In [ ]:
# ---- SETTINGS: change these per job
BLEND_NAME = None      # None = first .blend found in the inputs
SAMPLES    = 20        # 40 = same quality as my PC renders
WIDTH      = 720       # 720 -> 720x1280. Use 1080 for final 1080x1920
START, END = 473, 945     # frame range (30 frames = 1 second)
FPS        = 30

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
# Blender needs a few X/GL libraries even when rendering without a screen
apt-get -qq update > /dev/null && apt-get -qq install -y libxrender1 libxxf86vm1 libxfixes3 libxi6 libxkbcommon0 libsm6 libgl1 libegl1 > /dev/null
cd /kaggle/working
if [ ! -d blender ]; then
  wget -q https://download.blender.org/release/Blender5.2/blender-5.2.2-linux-x64.tar.xz
  tar xf blender-5.2.2-linux-x64.tar.xz && mv blender-5.2.2-linux-x64 blender && rm blender-5.2.2-linux-x64.tar.xz
fi
./blender/blender --version | head -1

In [ ]:
import glob, os, shutil
# the scene is uploaded in <10 MB parts (*.blend.part0, part1, ...): join them back into one .blend
parts = sorted(glob.glob('/kaggle/input/**/*.blend.part*', recursive=True), key=lambda p: int(p.rsplit('part', 1)[1]))  # part10 after part9
if parts:
    name = os.path.basename(parts[0]).split('.part')[0]
    with open('/kaggle/working/' + name, 'wb') as out:
        for p in parts: out.write(open(p, 'rb').read())
    print('joined', len(parts), 'parts ->', name, os.path.getsize('/kaggle/working/' + name), 'bytes')
blends = sorted(glob.glob('/kaggle/input/**/*.blend', recursive=True) + glob.glob('/kaggle/working/*.blend'))
print(blends)
BLEND = next(b for b in blends if BLEND_NAME is None or os.path.basename(b) == BLEND_NAME)
for f in ('post.py',):
    shutil.copy(glob.glob(f'/kaggle/input/**/{f}', recursive=True)[0], '/kaggle/working/' + f)
print('using', BLEND)

In [ ]:
%%writefile /kaggle/working/kaggle_render.py
# Render a packed .blend on whatever GPU the machine has (Kaggle: 2x Tesla T4), CPU fallback.
# Run: blender -b scene.blend -P kaggle_render.py -- <out_dir> <samples> <width> <start> <end>
import bpy, os, sys, time

argv = sys.argv[sys.argv.index("--") + 1:]
OUT = os.path.abspath(argv[0])
SAMPLES = int(argv[1]); WIDTH = int(argv[2]); START = int(argv[3]); END = int(argv[4])
os.makedirs(OUT, exist_ok=True)

sc = bpy.context.scene
r = sc.render
r.engine = "CYCLES"
sc.cycles.samples = SAMPLES
sc.cycles.use_denoising = True
r.resolution_x = WIDTH
r.resolution_y = int(WIDTH * 16 / 9) // 2 * 2
r.resolution_percentage = 100
r.image_settings.file_format = "PNG"

# ---- use every GPU we can find (OptiX is fastest on NVIDIA, then CUDA)
prefs = bpy.context.preferences.addons["cycles"].preferences
used = None
for backend in ("OPTIX", "CUDA"):
    try:
        prefs.compute_device_type = backend
        prefs.refresh_devices()
        gpus = [d for d in prefs.devices if d.type == backend]
        if gpus:
            for d in prefs.devices:
                d.use = d.type == backend          # GPUs only; CPU would slow the GPUs down
            used = backend
            print("GPU devices:", [d.name for d in gpus], "via", backend, flush=True)
            break
    except Exception as e:
        print("backend", backend, "not available:", e, flush=True)
if used:
    sc.cycles.device = "GPU"
else:
    sc.cycles.device = "CPU"
    print("No GPU found, rendering on CPU", flush=True)
# Kaggle's T4 driver can't create the OptiX denoiser -> always use OpenImageDenoise
sc.cycles.denoiser = "OPENIMAGEDENOISE"
try: sc.cycles.denoising_use_gpu = True
except Exception: pass

def render_to(out):
    r.filepath = out
    bpy.ops.render.render(write_still=True)

t_all = time.time()
for f in range(START, END + 1):
    out = os.path.join(OUT, f"frame_{f:04d}.png")
    if os.path.exists(out):                       # resume after a session cut-off
        continue
    t = time.time()
    sc.frame_set(f)
    try:
        render_to(out)
    except Exception as e:                        # OptiX trouble -> fall back to CUDA, then CPU
        print("render failed:", e, flush=True)
        for backend in ("CUDA", None):
            try:
                if backend:
                    prefs.compute_device_type = backend; prefs.refresh_devices()
                    for d in prefs.devices: d.use = d.type == backend
                    sc.cycles.device = "GPU"
                else:
                    sc.cycles.device = "CPU"
                try: sc.cycles.denoising_use_gpu = False
                except Exception: pass
                render_to(out); print("switched to", backend or "CPU", flush=True); break
            except Exception as e2:
                print("fallback", backend, "failed:", e2, flush=True)
        else:
            raise
    print(f"FRAME {f}/{END}  {time.time() - t:.1f}s", flush=True)
print(f"DONE {END - START + 1} frames in {(time.time() - t_all) / 60:.1f} min", flush=True)


In [ ]:
# ---- RENDER (prints one line per frame with its time)
!cd /kaggle/working && ./blender/blender -b "{BLEND}" -P kaggle_render.py -- frames {SAMPLES} {WIDTH} {START} {END} 2>&1 | grep -E 'GPU devices|No GPU|FRAME|DONE|Error'

In [ ]:
# ---- camera look (soft lens, glow, vignette, grain) + MP4
import subprocess, glob, os
assert glob.glob('/kaggle/working/frames/frame_*.png'), 'No frames rendered - check the render cell output above'
os.makedirs('/kaggle/working/post', exist_ok=True)
for f in sorted(glob.glob('/kaggle/working/frames/frame_*.png')):
    subprocess.run(['python', '/kaggle/working/post.py', f, '/kaggle/working/post/' + os.path.basename(f)], check=True, capture_output=True)
!ffmpeg -loglevel error -y -framerate {FPS} -start_number {START} -i /kaggle/working/post/frame_%04d.png -c:v libx264 -pix_fmt yuv420p -crf 17 /kaggle/working/doc_render.mp4
!cd /kaggle/working && zip -qr frames.zip frames && ls -lh doc_render.mp4 frames.zip

In [ ]:
# ---- clean up so the Output tab only shows the results
!rm -rf /kaggle/working/blender /kaggle/working/post /kaggle/working/*.blend
!ls -lh /kaggle/working